# Model Comparison & Final Analysis

Compare all recommendation algorithms on:
- **RMSE / MAE** (rating prediction accuracy)
- **Precision@K / Recall@K / NDCG@K** (ranking quality)
- **Coverage** (catalogue breadth)
- **Training time** (efficiency)

Models compared:
1. User-Based Collaborative Filtering
2. Item-Based Collaborative Filtering
3. SVD Matrix Factorization
4. Content-Based Filtering (genre features)
5. Ensemble (CF + SVD combined)

In [ ]:
import sys
import time
sys.path.append('../src')
sys.path.append('../evaluation')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split

from data_loader import MovieLensDataLoader
from models.collaborative import UserBasedCF, ItemBasedCF
from models.matrix_factorization import SVDRecommender
from models.content_based import ContentBasedRecommender
from metrics import rmse, mae, precision_at_k, recall_at_k, ndcg_at_k, coverage

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('Libraries loaded ✅')

## 1. Load Data

In [ ]:
loader = MovieLensDataLoader()
ratings, movies, users = loader.load_data()
genre_matrix = loader.get_genre_features()
movie_titles = loader.get_movie_titles()

print(f'Ratings:  {len(ratings):,}')
print(f'Users:    {ratings["user_id"].nunique():,}')
print(f'Movies:   {ratings["item_id"].nunique():,}')
print(f'Genres:   {genre_matrix.shape[1]}')

## 2. Train / Test Split

In [ ]:
train_ratings, test_ratings = train_test_split(
    ratings, test_size=0.2, random_state=42
)

train_matrix = train_ratings.pivot_table(
    index='user_id', columns='item_id', values='rating'
)

print(f'Train: {len(train_ratings):,} | Test: {len(test_ratings):,}')
print(f'Matrix: {train_matrix.shape}')

## 3. Train All Models

In [ ]:
models = {}
train_times = {}

# User-Based CF
t = time.time()
models['User CF'] = UserBasedCF(n_neighbors=15).fit(train_matrix)
train_times['User CF'] = time.time() - t
print(f'User CF trained in {train_times["User CF"]:.2f}s')

# Item-Based CF
t = time.time()
models['Item CF'] = ItemBasedCF(n_neighbors=15).fit(train_matrix)
train_times['Item CF'] = time.time() - t
print(f'Item CF trained in {train_times["Item CF"]:.2f}s')

# SVD
t = time.time()
models['SVD'] = SVDRecommender(n_factors=50).fit(train_matrix)
train_times['SVD'] = time.time() - t
print(f'SVD trained in {train_times["SVD"]:.2f}s')

# Content-Based
t = time.time()
models['Content-Based'] = ContentBasedRecommender(min_rating_threshold=4.0).fit(
    train_ratings, genre_matrix
)
train_times['Content-Based'] = time.time() - t
print(f'Content-Based trained in {train_times["Content-Based"]:.2f}s')

## 4. Rating Prediction Accuracy (RMSE / MAE)

In [ ]:
# Evaluate on a sample of test pairs (full test set is too slow for CF)
SAMPLE = 500
sample_test = test_ratings.sample(min(SAMPLE, len(test_ratings)), random_state=42)

results = {}
for name, model in models.items():
    preds, actuals = [], []
    for _, row in sample_test.iterrows():
        uid, iid, r = int(row['user_id']), int(row['item_id']), row['rating']
        preds.append(model.predict(uid, iid))
        actuals.append(r)
    results[name] = {
        'RMSE': rmse(actuals, preds),
        'MAE':  mae(actuals, preds),
    }
    print(f'{name:18s}  RMSE={results[name]["RMSE"]:.4f}  MAE={results[name]["MAE"]:.4f}')

## 5. Top-N Recommendation Quality (Precision / Recall / NDCG @10)

In [ ]:
K = 10
EVAL_USERS = 50  # Evaluate on 50 random users
THRESHOLD  = 4   # A rating ≥ 4 is considered 'relevant'

# Build ground-truth: for each user in test set, items they rated ≥ threshold
relevant_items = (
    test_ratings[test_ratings['rating'] >= THRESHOLD]
    .groupby('user_id')['item_id'].apply(set).to_dict()
)
# Pick users who have at least one relevant item in test AND are in train
eval_users = [
    u for u in list(relevant_items.keys())
    if u in train_matrix.index and len(relevant_items[u]) >= 1
][:EVAL_USERS]

topn_results = {name: {'P@K': [], 'R@K': [], 'NDCG@K': []} for name in models}

for uid in eval_users:
    rated = set(train_matrix.loc[uid].dropna().index)
    ground = relevant_items.get(uid, set())

    for name, model in models.items():
        if name == 'Content-Based':
            recs = model.recommend(uid, n_items=K, rated_items=rated)
        else:
            recs = model.recommend(uid, n_items=K)

        topn_results[name]['P@K'].append(precision_at_k(recs, ground, K))
        topn_results[name]['R@K'].append(recall_at_k(recs, ground, K))
        topn_results[name]['NDCG@K'].append(ndcg_at_k(recs, ground, K))

for name in topn_results:
    p = np.mean(topn_results[name]['P@K'])
    r = np.mean(topn_results[name]['R@K'])
    n = np.mean(topn_results[name]['NDCG@K'])
    results[name].update({'P@10': p, 'R@10': r, 'NDCG@10': n})
    print(f'{name:18s}  P@10={p:.4f}  R@10={r:.4f}  NDCG@10={n:.4f}')

## 6. Coverage

In [ ]:
n_items_total = ratings['item_id'].nunique()
COVERAGE_USERS = 100
cov_users = [u for u in train_matrix.index[:COVERAGE_USERS] if u in train_matrix.index]

for name, model in models.items():
    all_recs = []
    for uid in cov_users:
        rated = set(train_matrix.loc[uid].dropna().index)
        if name == 'Content-Based':
            recs = model.recommend(uid, n_items=10, rated_items=rated)
        else:
            recs = model.recommend(uid, n_items=10)
        if recs:
            all_recs.append(recs)
    cov = coverage(all_recs, n_items_total)
    results[name]['Coverage'] = cov
    print(f'{name:18s}  Coverage={cov:.4f}')

## 7. Results Summary Table

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary.index.name = 'Model'
print('\n=== Model Comparison Summary ===')
print(summary.to_string())

## 8. Visualisations

In [ ]:
model_names = list(results.keys())
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Recommendation Model Comparison — MovieLens 100K', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('RMSE',     'lower is better', True),
    ('MAE',      'lower is better', True),
    ('P@10',     'higher is better', False),
    ('R@10',     'higher is better', False),
    ('NDCG@10',  'higher is better', False),
    ('Coverage', 'higher is better', False),
]

for ax, (metric, caption, lower_is_better) in zip(axes.flat, metrics_to_plot):
    vals = [results[m].get(metric, 0) for m in model_names]
    bars = ax.bar(model_names, vals, color=colors)
    ax.set_title(f'{metric}  ({caption})', fontsize=11)
    ax.set_ylabel(metric)
    ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5)
    # Highlight best
    best_idx = int(np.argmin(vals) if lower_is_better else np.argmax(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to data/processed/model_comparison.png')

## 9. Training Time Comparison

In [ ]:
plt.figure(figsize=(8, 4))
bars = plt.bar(list(train_times.keys()), list(train_times.values()), color=colors)
plt.title('Training Time per Model (seconds)', fontsize=13)
plt.ylabel('Time (s)')
plt.xticks(rotation=15, ha='right')
for bar, val in zip(bars, train_times.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}s', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 10. Sample Recommendations with Movie Titles

In [ ]:
DEMO_USER = 1
rated_by_user = set(train_matrix.loc[DEMO_USER].dropna().index)

print(f'\n=== Top-10 Recommendations for User {DEMO_USER} ===\n')

for name, model in models.items():
    if name == 'Content-Based':
        recs = model.recommend(DEMO_USER, n_items=10, rated_items=rated_by_user)
    else:
        recs = model.recommend(DEMO_USER, n_items=10)

    df = loader.get_recommendations_df(recs, user_id=DEMO_USER)
    print(f'--- {name} ---')
    print(df[['rank', 'title', 'genres']].to_string(index=False))
    print()

## 11. Content-Based: Similar Movies

In [ ]:
# Find movies most similar to a seed movie by genre content
SEED_MOVIE_ID = 50  # Star Wars (1977)
seed_title = movie_titles.get(SEED_MOVIE_ID, 'Unknown')
similar = models['Content-Based'].get_similar_items(SEED_MOVIE_ID, n_items=10)

print(f'Movies most similar to: "{seed_title}" (id={SEED_MOVIE_ID})')
print(f'{"Rank":>5}  {"Item ID":>8}  {"Similarity":>10}  Title')
print('-' * 65)
for rank, (iid, sim) in enumerate(similar, start=1):
    title = movie_titles.get(iid, 'Unknown')
    print(f'{rank:>5}  {iid:>8}  {sim:>10.4f}  {title}')

## 12. Radar Chart: Overall Model Profiles

In [ ]:
from matplotlib.patches import FancyArrowPatch

# Normalise metrics to [0, 1] for radar chart.
# RMSE/MAE are inverted (lower = better → score = 1 - normalised)
radar_metrics = ['RMSE', 'MAE', 'P@10', 'R@10', 'NDCG@10', 'Coverage']
invert = {'RMSE', 'MAE'}

raw = {m: [results[name].get(m, 0) for name in model_names] for m in radar_metrics}
norm = {}
for m, vals in raw.items():
    lo, hi = min(vals), max(vals)
    if hi == lo:
        norm[m] = [0.5] * len(vals)
    elif m in invert:
        norm[m] = [1 - (v - lo)/(hi - lo) for v in vals]
    else:
        norm[m] = [(v - lo)/(hi - lo) for v in vals]

N = len(radar_metrics)
angles = [n * 2 * np.pi / N for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})
for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [norm[m][i] for m in radar_metrics] + [norm[radar_metrics[0]][i]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Model Profile Radar (normalised)', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig('../data/processed/radar_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Radar chart saved.')